**Capítulo 15 – Procesamiento de secuencias con RNNs (versión recortada en español)**

_Este notebook contiene una selección del código del capítulo 15 de_ Hands-On Machine Learning with Scikit-Learn, Keras and TensorFlow _(3ª edición) de Aurélien Géron, enfocada en RNN simples, RNN profundas, LSTM y GRU._

> **Traducción al español y versión recortada.** Esta libreta es una adaptación de
> [`15_processing_sequences_using_rnns_and_cnns.ipynb`](https://github.com/ageron/handson-ml3/blob/main/15_processing_sequences_using_rnns_and_cnns.ipynb)
> del repositorio [ageron/handson-ml3](https://github.com/ageron/handson-ml3) de Aurélien Géron, distribuido bajo la Licencia Apache 2.0.
> Se tradujo al español y se recortó para uso en clase. La lista de cambios, la atribución y la licencia completa están al final de esta libreta.

## Qué vas a hacer en este notebook

1. Explorar una serie de tiempo real: pasajeros diarios del transporte público de Chicago.
2. Construir un **baseline ingenuo** y un modelo lineal para tener contra qué comparar.
3. Entrenar una **RNN simple** y una **RNN profunda** para pronosticar el día siguiente.
4. Entrenar una **LSTM** y una **GRU** para pronosticar los próximos 14 días.

**Recomendación:** ejecuta este notebook en Google Colab con GPU (*Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU*). En CPU algunos entrenamientos pueden tardar varios minutos.

**Importante:** ejecuta las celdas en orden. Las secciones de LSTM y GRU usan datos que se preparan en secciones anteriores.

# Configuración

Este proyecto requiere Python 3.7 o superior:

In [ ]:
import sys

assert sys.version_info >= (3, 7)

Y TensorFlow ≥ 2.8:

In [ ]:
from packaging import version
import tensorflow as tf

assert version.parse(tf.__version__) >= version.parse("2.8.0")

Definimos tamaños de letra por defecto para que las figuras se vean mejor:

In [ ]:
import matplotlib.pyplot as plt

plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

Creamos la carpeta `images/rnn` (si no existe) y definimos la función `save_fig()`, que guarda las figuras en alta resolución:

In [ ]:
from pathlib import Path

IMAGES_PATH = Path() / "images" / "rnn"
IMAGES_PATH.mkdir(parents=True, exist_ok=True)

def save_fig(fig_id, tight_layout=True, fig_extension="png", resolution=300):
    path = IMAGES_PATH / f"{fig_id}.{fig_extension}"
    if tight_layout:
        plt.tight_layout()
    plt.savefig(path, format=fig_extension, dpi=resolution)

Este capítulo puede ser muy lento sin GPU, así que verificamos si hay una y, si no, mostramos una advertencia:

In [ ]:
if not tf.config.list_physical_devices('GPU'):
    print("No se detectó GPU. Las redes neuronales pueden ser muy lentas sin GPU.")
    if "google.colab" in sys.modules:
        print("Ve a Entorno de ejecución > Cambiar tipo de entorno de ejecución "
              "y selecciona un acelerador GPU.")
    if "kaggle_secrets" in sys.modules:
        print("Ve a Settings > Accelerator y selecciona GPU.")

# RNNs básicas

## El dataset: pasajeros del transporte público de Chicago

Descargamos los datos de pasajeros desde el proyecto ageron/data. Originalmente provienen de la Autoridad de Tránsito de Chicago (CTA) y se descargaron del [portal de datos de Chicago](https://homl.info/ridership).

**Nota:** en versiones recientes de Keras, `get_file()` coloca el directorio extraído dentro de otro directorio cuyo nombre termina en `_extracted`, por eso el siguiente código revisa ese caso:

In [ ]:
filepath = tf.keras.utils.get_file(
    "ridership.tgz",
    "https://github.com/ageron/data/raw/main/ridership.tgz",
    cache_dir=".",
    extract=True
)
if "_extracted" in filepath:
    ridership_path = Path(filepath) / "ridership"
else:
    ridership_path = Path(filepath).with_name("ridership")

In [ ]:
import pandas as pd
from pathlib import Path

path = ridership_path / "CTA_-_Ridership_-_Daily_Boarding_Totals.csv"
df = pd.read_csv(path, parse_dates=["service_date"])
df.columns = ["date", "day_type", "bus", "rail", "total"]  # nombres más cortos
df = df.sort_values("date").set_index("date")
df = df.drop("total", axis=1)  # no necesitamos el total, es solo bus + rail
df = df.drop_duplicates()  # elimina meses duplicados (2011-10 y 2014-07)

Cada fila es un día. Las columnas son:

- `day_type`: tipo de día (`W` = entre semana, `A` = sábado, `U` = domingo o día festivo).
- `bus`: número de pasajeros en autobús ese día.
- `rail`: número de pasajeros en tren ese día.

In [ ]:
df.head()

Veamos los primeros meses de 2019 (Pandas incluye ambos extremos del rango):

In [ ]:
import matplotlib.pyplot as plt

df["2019-03":"2019-05"].plot(grid=True, marker=".", figsize=(8, 3.5))
save_fig("daily_ridership_plot")  # código extra – guarda la figura
plt.show()

Se ve un patrón semanal muy claro: muchos pasajeros entre semana y pocos el fin de semana.
Eso sugiere un pronóstico muy simple: **predecir que mañana habrá los mismos pasajeros que hace una semana**.
Esto se llama *pronóstico ingenuo* (*naive forecast*) y será nuestro baseline.

La siguiente gráfica compara la serie original con la serie desplazada 7 días (línea punteada), y abajo muestra la diferencia entre ambas:

In [ ]:
diff_7 = df[["bus", "rail"]].diff(7)["2019-03":"2019-05"]

fig, axs = plt.subplots(2, 1, sharex=True, figsize=(8, 5))
df.plot(ax=axs[0], legend=False, marker=".")  # serie original
df.shift(7).plot(ax=axs[0], grid=True, legend=False, linestyle=":")  # serie desplazada 7 días
diff_7.plot(ax=axs[1], grid=True, marker=".")  # diferencia de 7 días
axs[0].set_ylim([170_000, 900_000])  # código extra – mejora la gráfica
save_fig("differencing_plot")  # código extra – guarda la figura
plt.show()

Los picos grandes en la diferencia suelen coincidir con días festivos. Por ejemplo, el 27 de mayo de 2019 fue Memorial Day, un lunes festivo:

In [ ]:
list(df.loc["2019-05-25":"2019-05-27"]["day_type"])

Error absoluto medio (MAE, *mean absolute error*), también llamado desviación absoluta media (MAD), del pronóstico ingenuo:

In [ ]:
diff_7.abs().mean()

Error absoluto porcentual medio (MAPE, *mean absolute percentage error*):

In [ ]:
targets = df[["bus", "rail"]]["2019-03":"2019-05"]
(diff_7 / targets).abs().mean()

**Guarda estos números.** Cualquier modelo que entrenemos debería tener un MAE menor que el pronóstico ingenuo; si no, no vale la pena.

Ahora veamos la estacionalidad anual y las tendencias de largo plazo:

**Nota:** la API de Pandas para calcular la media mensual ha cambiado varias veces, así que el código intenta primero la API más nueva y, si falla, usa versiones anteriores:

In [ ]:
period = slice("2001", "2019")
try:
    df_monthly = df.select_dtypes(include="number").resample('ME').mean()  # media de cada mes
    rolling_average_12_months = df_monthly.loc[period].rolling(window=12).mean()
except ValueError as ex:
    try:
        df_monthly = df.select_dtypes(include="number").resample('M').mean()  # media de cada mes
        rolling_average_12_months = df_monthly.loc[period].rolling(window=12).mean()
    except ValueError as ex:
        df_monthly = df.resample('M').mean()  # media de cada mes
        rolling_average_12_months = df_monthly[period].rolling(window=12).mean()

fig, ax = plt.subplots(figsize=(8, 4))
df_monthly[period].plot(ax=ax, marker=".")
rolling_average_12_months.plot(ax=ax, grid=True, legend=False)
save_fig("long_term_ridership_plot")  # código extra – guarda la figura
plt.show()

In [ ]:
df_monthly.diff(12)[period].plot(grid=True, marker=".", figsize=(8, 3))
save_fig("yearly_diff_plot")  # código extra – guarda la figura
plt.show()

## Preparar los datos para modelos de machine learning

Una red neuronal no recibe la serie completa, sino **ventanas**: secuencias de valores consecutivos como entrada,
y el valor siguiente como objetivo. Keras tiene la función `timeseries_dataset_from_array()` para construirlas.

En este ejemplo de juguete, cada entrada tiene 3 valores consecutivos y el objetivo es el valor que sigue:

In [ ]:
import tensorflow as tf

my_series = [0, 1, 2, 3, 4, 5]
my_dataset = tf.keras.utils.timeseries_dataset_from_array(
    my_series,
    targets=my_series[3:],  # los objetivos están 3 pasos en el futuro
    sequence_length=3,
    batch_size=2
)
list(my_dataset)

Otra forma de crear ventanas es con el método `window()` de `tf.data.Dataset`.
Este método devuelve un dataset de *datasets* (cada ventana es a su vez un dataset):

In [ ]:
for window_dataset in tf.data.Dataset.range(6).window(4, shift=1):
    for element in window_dataset:
        print(f"{element}", end=" ")
    print()

Con `drop_remainder=True` descartamos las ventanas incompletas, y con `flat_map()` convertimos cada ventana en un tensor:

In [ ]:
dataset = tf.data.Dataset.range(6).window(4, shift=1, drop_remainder=True)
dataset = dataset.flat_map(lambda window_dataset: window_dataset.batch(4))
for window_tensor in dataset:
    print(f"{window_tensor}")

Empaquetamos esa idea en una función auxiliar, `to_windows()`. **La usaremos más adelante** para preparar los datos de LSTM y GRU:

In [ ]:
def to_windows(dataset, length):
    dataset = dataset.window(length, shift=1, drop_remainder=True)
    return dataset.flat_map(lambda window_ds: window_ds.batch(length))

Por ejemplo, separamos cada ventana en entrada (los primeros valores) y objetivo (el último):

In [ ]:
dataset = to_windows(tf.data.Dataset.range(6), 4)
dataset = dataset.map(lambda window: (window[:-1], window[-1]))
list(dataset.batch(2))

Antes de seguir, dividimos la serie en tres periodos: entrenamiento, validación y prueba. Por ahora no veremos los datos de prueba.

Dividimos entre un millón para que los valores estén cerca de 1, lo que facilita el entrenamiento:

In [ ]:
rail_train = df["rail"]["2016-01":"2018-12"] / 1e6
rail_valid = df["rail"]["2019-01":"2019-05"] / 1e6
rail_test = df["rail"]["2019-06":] / 1e6

Creamos los datasets de entrenamiento y validación con ventanas de 56 días (8 semanas). El objetivo es el número de pasajeros de tren del día siguiente:

In [ ]:
seq_length = 56
tf.random.set_seed(42)  # código extra – asegura reproducibilidad
train_ds = tf.keras.utils.timeseries_dataset_from_array(
    rail_train.to_numpy(),
    targets=rail_train[seq_length:],
    sequence_length=seq_length,
    batch_size=32,
    shuffle=True,
    seed=42
)
valid_ds = tf.keras.utils.timeseries_dataset_from_array(
    rail_valid.to_numpy(),
    targets=rail_valid[seq_length:],
    sequence_length=seq_length,
    batch_size=32
)

## Pronóstico con un modelo lineal

Antes de usar una RNN, probamos un modelo lineal simple (una sola neurona densa). Es otro punto de comparación:

In [ ]:
tf.random.set_seed(42)
model = tf.keras.Sequential([
    tf.keras.layers.Dense(1, input_shape=[seq_length])
])
early_stopping_cb = tf.keras.callbacks.EarlyStopping(
    monitor="val_mae", patience=50, restore_best_weights=True)
opt = tf.keras.optimizers.SGD(learning_rate=0.02, momentum=0.9)
model.compile(loss=tf.keras.losses.Huber(), optimizer=opt, metrics=["mae"])
history = model.fit(train_ds, validation_data=valid_ds, epochs=500,
                    callbacks=[early_stopping_cb])

In [ ]:
# código extra – evalúa el modelo
valid_loss, valid_mae = model.evaluate(valid_ds)
valid_mae * 1e6

## Usando una RNN simple

La RNN más sencilla posible: **una sola neurona recurrente**. `input_shape=[None, 1]` significa que la secuencia
puede tener cualquier longitud (`None`) y que en cada paso de tiempo hay una sola característica (pasajeros de tren).

In [ ]:
tf.random.set_seed(42)  # código extra – asegura reproducibilidad
model = tf.keras.Sequential([
    tf.keras.layers.SimpleRNN(1, input_shape=[None, 1])
])

Definimos una función auxiliar para compilar, entrenar y evaluar los modelos. La reutilizaremos varias veces. Devuelve el MAE de validación en número de pasajeros:

In [ ]:
# código extra – define una función auxiliar que reutilizaremos varias veces

def fit_and_evaluate(model, train_set, valid_set, learning_rate, epochs=500):
    early_stopping_cb = tf.keras.callbacks.EarlyStopping(
        monitor="val_mae", patience=50, restore_best_weights=True)
    opt = tf.keras.optimizers.SGD(learning_rate=learning_rate, momentum=0.9)
    model.compile(loss=tf.keras.losses.Huber(), optimizer=opt, metrics=["mae"])
    history = model.fit(train_set, validation_data=valid_set, epochs=epochs,
                        callbacks=[early_stopping_cb])
    valid_loss, valid_mae = model.evaluate(valid_set)
    return valid_mae * 1e6

In [ ]:
fit_and_evaluate(model, train_ds, valid_ds, learning_rate=0.02)

Una sola neurona es muy poco: su salida está limitada al rango de la tangente hiperbólica (−1 a 1) y tiene muy pocos parámetros.
Probemos con una capa recurrente de 32 neuronas seguida de una capa densa de salida:

In [ ]:
tf.random.set_seed(42)  # código extra – asegura reproducibilidad
univar_model = tf.keras.Sequential([
    tf.keras.layers.SimpleRNN(32, input_shape=[None, 1]),
    tf.keras.layers.Dense(1)  # sin función de activación por defecto
])

In [ ]:
# código extra – compila, entrena y evalúa el modelo, igual que antes
fit_and_evaluate(univar_model, train_ds, valid_ds, learning_rate=0.05)

## RNNs profundas

Apilamos tres capas recurrentes. Las capas intermedias necesitan `return_sequences=True` para entregar
su salida en **todos** los pasos de tiempo a la siguiente capa recurrente, no solo en el último.

In [ ]:
tf.random.set_seed(42)  # código extra – asegura reproducibilidad
deep_model = tf.keras.Sequential([
    tf.keras.layers.SimpleRNN(32, return_sequences=True, input_shape=[None, 1]),
    tf.keras.layers.SimpleRNN(32, return_sequences=True),
    tf.keras.layers.SimpleRNN(32),
    tf.keras.layers.Dense(1)
])

In [ ]:
# código extra – compila, entrena y evalúa el modelo, igual que antes
fit_and_evaluate(deep_model, train_ds, valid_ds, learning_rate=0.01)

# Preparación de datos para LSTM y GRU

Para LSTM y GRU vamos a resolver un problema más difícil, con dos cambios respecto a lo anterior:

1. **Entrada multivariada:** en cada día el modelo recibe 5 características: pasajeros de autobús, pasajeros de tren
   y el tipo de día de *mañana* codificado en one-hot (3 columnas). Conocemos el tipo de día de mañana de antemano, así que es válido usarlo.
2. **Salida de secuencia a secuencia (*seq2seq*):** en cada paso de tiempo el modelo pronostica los **próximos 14 días** de pasajeros de tren.

Estas celdas solo preparan los datos; no hay que entrenar nada todavía.

**Nota:** en versiones recientes de Pandas, `get_dummies()` devuelve booleanos en lugar de flotantes, por eso agregamos `dtype=float`.

In [ ]:
df_mulvar = df[["bus", "rail"]] / 1e6  # usamos las series de autobús y tren como entrada
df_mulvar["next_day_type"] = df["day_type"].shift(-1)  # conocemos el tipo de día de mañana
df_mulvar = pd.get_dummies(df_mulvar, dtype=float)  # codificación one-hot del tipo de día

In [ ]:
mulvar_train = df_mulvar["2016-01":"2018-12"]
mulvar_valid = df_mulvar["2019-01":"2019-05"]
mulvar_test = df_mulvar["2019-06":]

In [ ]:
mulvar_train.head()

Ahora queremos que en el paso de tiempo 0 el modelo pronostique los pasos 1 a 14, en el paso 1 pronostique los pasos 2 a 15, y así sucesivamente.
Al final de la secuencia de 56 días pronostica los días 57 a 70. El modelo es *causal*: en cada paso solo ve el pasado.

Para preparar los datos podemos usar `to_windows()` dos veces, y así obtener secuencias de ventanas consecutivas:

In [ ]:
my_series = tf.data.Dataset.range(7)
dataset = to_windows(to_windows(my_series, 3), 4)
list(dataset)

Luego separamos cada elemento en entradas (primer valor de cada ventana) y objetivos (el resto de cada ventana):

In [ ]:
dataset = dataset.map(lambda S: (S[:, 0], S[:, 1:]))
list(dataset)

Empaquetamos esta idea en una función que además se encarga de mezclar (opcional) y agrupar en lotes. `target_col=1` indica que el objetivo es la columna `rail`:

In [ ]:
def to_seq2seq_dataset(series, seq_length=56, ahead=14, target_col=1,
                       batch_size=32, shuffle=False, seed=None):
    ds = to_windows(tf.data.Dataset.from_tensor_slices(series), ahead + 1)
    ds = to_windows(ds, seq_length).map(
        lambda S: (S[:, 0], S[:, 1:, target_col]))
    if shuffle:
        ds = ds.shuffle(8 * batch_size, seed=seed)
    return ds.batch(batch_size)

In [ ]:
seq2seq_train = to_seq2seq_dataset(mulvar_train, shuffle=True, seed=42)
seq2seq_valid = to_seq2seq_dataset(mulvar_valid)

### Referencia: RNN simple seq2seq

Para tener un punto de comparación, entrenamos primero una RNN simple con estos mismos datos.
Como la salida es una secuencia, la capa recurrente usa `return_sequences=True`, y la capa `Dense(14)` se aplica en cada paso de tiempo.

In [ ]:
tf.random.set_seed(42)  # código extra – asegura reproducibilidad
seq2seq_model = tf.keras.Sequential([
    tf.keras.layers.SimpleRNN(32, return_sequences=True, input_shape=[None, 5]),
    tf.keras.layers.Dense(14)
    # equivalente: tf.keras.layers.TimeDistributed(tf.keras.layers.Dense(14))
    # también equivalente: tf.keras.layers.Conv1D(14, kernel_size=1)
])

In [ ]:
fit_and_evaluate(seq2seq_model, seq2seq_train, seq2seq_valid,
                 learning_rate=0.1)

# LSTMs

Para usar una LSTM basta con cambiar `SimpleRNN` por `LSTM`; el resto del modelo es igual.
Internamente, cada celda LSTM tiene compuertas de olvido, entrada y salida, y un estado de largo plazo que le permite recordar información por más pasos de tiempo.

In [ ]:
tf.random.set_seed(42)  # código extra – asegura reproducibilidad
lstm_model = tf.keras.models.Sequential([
    tf.keras.layers.LSTM(32, return_sequences=True, input_shape=[None, 5]),
    tf.keras.layers.Dense(14)
])

Entrenamos solo 5 épocas para mostrar que funciona (puedes aumentarlas si quieres):

In [ ]:
fit_and_evaluate(lstm_model, seq2seq_train, seq2seq_valid,
                 learning_rate=0.1, epochs=5)

# GRUs

La GRU es una versión simplificada de la LSTM: combina estados y usa menos compuertas, por lo que tiene menos parámetros
y suele entrenar más rápido. De nuevo, solo cambiamos el tipo de capa:

In [ ]:
tf.random.set_seed(42)  # código extra – asegura reproducibilidad
gru_model = tf.keras.Sequential([
    tf.keras.layers.GRU(32, return_sequences=True, input_shape=[None, 5]),
    tf.keras.layers.Dense(14)
])

Entrenamos solo 5 épocas para mostrar que funciona (puedes aumentarlas si quieres):

In [ ]:
fit_and_evaluate(gru_model, seq2seq_train, seq2seq_valid,
                 learning_rate=0.1, epochs=5)

## Para pensar

- La RNN simple seq2seq se entrenó hasta 500 épocas con *early stopping*, mientras que la LSTM y la GRU solo 5. ¿Es justa la comparación?
  Prueba aumentar `epochs` en la LSTM y la GRU y compara de nuevo.
- Compara el número de parámetros de las tres capas recurrentes con `seq2seq_model.summary()`, `lstm_model.summary()` y `gru_model.summary()`.
  ¿Por qué la LSTM tiene aproximadamente 4 veces los parámetros de la RNN simple y la GRU aproximadamente 3 veces?

---

## Licencia y atribución

Esta libreta es una obra derivada de **15_processing_sequences_using_rnns_and_cnns.ipynb** de **[ageron/handson-ml3](https://github.com/ageron/handson-ml3)**, el repositorio complementario de *Hands-On Machine Learning with Scikit-Learn, Keras & TensorFlow* (3.ª edición) de Aurélien Géron (O'Reilly Media), y una traducción al español de dicho contenido.

**Cambios realizados respecto al archivo original** (Noé Sámano, 2026):

- Se tradujeron al español las celdas de texto y los comentarios del código.
- Se eliminaron las secciones de modelos ARIMA/SARIMA, pronóstico multivariado y de varios pasos con RNN simple, RNN profundas con Layer Normalization, la clase RNN personalizada, convoluciones 1D, WaveNet y las soluciones a los ejercicios.
- Se conservaron solo las celdas de preparación de datos multivariados y *seq2seq* necesarias para entrenar los modelos LSTM y GRU.
- Se agregaron notas explicativas propias, una introducción y una sección final de preguntas para reflexionar.

El texto del libro no está incluido; para la explicación completa consulta el capítulo 15 del libro.

El repositorio original se distribuye bajo la **Licencia Apache 2.0**, y esta libreta se distribuye bajo la misma licencia. A continuación se reproduce una copia literal de la licencia (en su idioma original).

```
                                 Apache License
                           Version 2.0, January 2004
                        http://www.apache.org/licenses/

   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION

   1. Definitions.

      "License" shall mean the terms and conditions for use, reproduction,
      and distribution as defined by Sections 1 through 9 of this document.

      "Licensor" shall mean the copyright owner or entity authorized by
      the copyright owner that is granting the License.

      "Legal Entity" shall mean the union of the acting entity and all
      other entities that control, are controlled by, or are under common
      control with that entity. For the purposes of this definition,
      "control" means (i) the power, direct or indirect, to cause the
      direction or management of such entity, whether by contract or
      otherwise, or (ii) ownership of fifty percent (50%) or more of the
      outstanding shares, or (iii) beneficial ownership of such entity.

      "You" (or "Your") shall mean an individual or Legal Entity
      exercising permissions granted by this License.

      "Source" form shall mean the preferred form for making modifications,
      including but not limited to software source code, documentation
      source, and configuration files.

      "Object" form shall mean any form resulting from mechanical
      transformation or translation of a Source form, including but
      not limited to compiled object code, generated documentation,
      and conversions to other media types.

      "Work" shall mean the work of authorship, whether in Source or
      Object form, made available under the License, as indicated by a
      copyright notice that is included in or attached to the work
      (an example is provided in the Appendix below).

      "Derivative Works" shall mean any work, whether in Source or Object
      form, that is based on (or derived from) the Work and for which the
      editorial revisions, annotations, elaborations, or other modifications
      represent, as a whole, an original work of authorship. For the purposes
      of this License, Derivative Works shall not include works that remain
      separable from, or merely link (or bind by name) to the interfaces of,
      the Work and Derivative Works thereof.

      "Contribution" shall mean any work of authorship, including
      the original version of the Work and any modifications or additions
      to that Work or Derivative Works thereof, that is intentionally
      submitted to Licensor for inclusion in the Work by the copyright owner
      or by an individual or Legal Entity authorized to submit on behalf of
      the copyright owner. For the purposes of this definition, "submitted"
      means any form of electronic, verbal, or written communication sent
      to the Licensor or its representatives, including but not limited to
      communication on electronic mailing lists, source code control systems,
      and issue tracking systems that are managed by, or on behalf of, the
      Licensor for the purpose of discussing and improving the Work, but
      excluding communication that is conspicuously marked or otherwise
      designated in writing by the copyright owner as "Not a Contribution."

      "Contributor" shall mean Licensor and any individual or Legal Entity
      on behalf of whom a Contribution has been received by Licensor and
      subsequently incorporated within the Work.

   2. Grant of Copyright License. Subject to the terms and conditions of
      this License, each Contributor hereby grants to You a perpetual,
      worldwide, non-exclusive, no-charge, royalty-free, irrevocable
      copyright license to reproduce, prepare Derivative Works of,
      publicly display, publicly perform, sublicense, and distribute the
      Work and such Derivative Works in Source or Object form.

   3. Grant of Patent License. Subject to the terms and conditions of
      this License, each Contributor hereby grants to You a perpetual,
      worldwide, non-exclusive, no-charge, royalty-free, irrevocable
      (except as stated in this section) patent license to make, have made,
      use, offer to sell, sell, import, and otherwise transfer the Work,
      where such license applies only to those patent claims licensable
      by such Contributor that are necessarily infringed by their
      Contribution(s) alone or by combination of their Contribution(s)
      with the Work to which such Contribution(s) was submitted. If You
      institute patent litigation against any entity (including a
      cross-claim or counterclaim in a lawsuit) alleging that the Work
      or a Contribution incorporated within the Work constitutes direct
      or contributory patent infringement, then any patent licenses
      granted to You under this License for that Work shall terminate
      as of the date such litigation is filed.

   4. Redistribution. You may reproduce and distribute copies of the
      Work or Derivative Works thereof in any medium, with or without
      modifications, and in Source or Object form, provided that You
      meet the following conditions:

      (a) You must give any other recipients of the Work or
          Derivative Works a copy of this License; and

      (b) You must cause any modified files to carry prominent notices
          stating that You changed the files; and

      (c) You must retain, in the Source form of any Derivative Works
          that You distribute, all copyright, patent, trademark, and
          attribution notices from the Source form of the Work,
          excluding those notices that do not pertain to any part of
          the Derivative Works; and

      (d) If the Work includes a "NOTICE" text file as part of its
          distribution, then any Derivative Works that You distribute must
          include a readable copy of the attribution notices contained
          within such NOTICE file, excluding those notices that do not
          pertain to any part of the Derivative Works, in at least one
          of the following places: within a NOTICE text file distributed
          as part of the Derivative Works; within the Source form or
          documentation, if provided along with the Derivative Works; or,
          within a display generated by the Derivative Works, if and
          wherever such third-party notices normally appear. The contents
          of the NOTICE file are for informational purposes only and
          do not modify the License. You may add Your own attribution
          notices within Derivative Works that You distribute, alongside
          or as an addendum to the NOTICE text from the Work, provided
          that such additional attribution notices cannot be construed
          as modifying the License.

      You may add Your own copyright statement to Your modifications and
      may provide additional or different license terms and conditions
      for use, reproduction, or distribution of Your modifications, or
      for any such Derivative Works as a whole, provided Your use,
      reproduction, and distribution of the Work otherwise complies with
      the conditions stated in this License.

   5. Submission of Contributions. Unless You explicitly state otherwise,
      any Contribution intentionally submitted for inclusion in the Work
      by You to the Licensor shall be under the terms and conditions of
      this License, without any additional terms or conditions.
      Notwithstanding the above, nothing herein shall supersede or modify
      the terms of any separate license agreement you may have executed
      with Licensor regarding such Contributions.

   6. Trademarks. This License does not grant permission to use the trade
      names, trademarks, service marks, or product names of the Licensor,
      except as required for reasonable and customary use in describing the
      origin of the Work and reproducing the content of the NOTICE file.

   7. Disclaimer of Warranty. Unless required by applicable law or
      agreed to in writing, Licensor provides the Work (and each
      Contributor provides its Contributions) on an "AS IS" BASIS,
      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or
      implied, including, without limitation, any warranties or conditions
      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A
      PARTICULAR PURPOSE. You are solely responsible for determining the
      appropriateness of using or redistributing the Work and assume any
      risks associated with Your exercise of permissions under this License.

   8. Limitation of Liability. In no event and under no legal theory,
      whether in tort (including negligence), contract, or otherwise,
      unless required by applicable law (such as deliberate and grossly
      negligent acts) or agreed to in writing, shall any Contributor be
      liable to You for damages, including any direct, indirect, special,
      incidental, or consequential damages of any character arising as a
      result of this License or out of the use or inability to use the
      Work (including but not limited to damages for loss of goodwill,
      work stoppage, computer failure or malfunction, or any and all
      other commercial damages or losses), even if such Contributor
      has been advised of the possibility of such damages.

   9. Accepting Warranty or Additional Liability. While redistributing
      the Work or Derivative Works thereof, You may choose to offer,
      and charge a fee for, acceptance of support, warranty, indemnity,
      or other liability obligations and/or rights consistent with this
      License. However, in accepting such obligations, You may act only
      on Your own behalf and on Your sole responsibility, not on behalf
      of any other Contributor, and only if You agree to indemnify,
      defend, and hold each Contributor harmless for any liability
      incurred by, or claims asserted against, such Contributor by reason
      of your accepting any such warranty or additional liability.

   END OF TERMS AND CONDITIONS
```